# Big Data Analytics - Book Recommendation with Implicit Feedback


Thomas Timmermans - 2365790

Milan Muroni - 2262576

In [2]:
%pip install scikit-learn implicit

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 560.1 kB/s eta 0:00:14
   -- ------------------------------------- 0.5/8.3 MB 560.1 kB/s eta 0:00:14
   --- ------------------------------------ 0.8/8.3 MB 685.3 kB/s eta 0:00:12
   ----- ---------------------------------- 1.0/8.3 MB 786.4 kB/s eta 0:00:10
   ----- ---------------------------------- 1.0/8.3 MB 786.4 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.3 MB 714.3 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.3 MB 714.3 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.3 MB 676.7 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.3 MB 676.7 kB/s eta 0:00:10
   -------- ----------


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import implicit

In [4]:
import json

file = "../datasets/books/Books.jsonl"
likes = []

with open(file, 'r') as fp:
    for i, line in enumerate(fp):
        data = json.loads(line.strip())
        # Alleen ratings strikt boven 4.0 tellen als 'like' (x_ui = 1)
        if data.get('rating', 0) > 4.0:
            likes.append((data['user_id'], data['parent_asin']))
        
print(f"Total likes: {len(likes)}")

df = pd.DataFrame(likes, columns=['user_id', 'book_id']).drop_duplicates()




Total likes: 20391478


## data inladen en filteren

In [5]:


print(f"Total unique likes: {len(df)}")

# 1. Filter op actieve users en boeken (bijv. minimaal 15 likes per user/boek)
# Pas '15' aan (bijv. naar 10 of 20) tot je totaal tussen de 100.000 en 300.000 likes zit
min_likes = 15
for _ in range(3):
    df = df[df['user_id'].map(df['user_id'].value_counts()) >= min_likes]
    df = df[df['book_id'].map(df['book_id'].value_counts()) >= min_likes]

print(f"Total likes after filtering: {len(df)}")

# 2. Zet tekst-ID's om naar simpele getallen (0, 1, 2, ...) voor de matrix
df['user_idx'] = df['user_id'].astype('category').cat.codes
df['book_idx'] = df['book_id'].astype('category').cat.codes

# 3. Bereken de statistieken (zoals gevraagd in de opdracht)
n_users = df['user_idx'].nunique()
n_books = df['book_idx'].nunique()
n_likes = len(df)
sparsity = 1 - (n_likes / (n_users * n_books))

print(f"Users: {n_users}")
print(f"Books: {n_books}")
print(f"Positive interactions: {n_likes}")
print(f"Sparsity: {sparsity:.4%}")

# 4. Maak de sparse matrix X (voorkomt een te grote dense matrix)
X = csr_matrix((np.ones(n_likes), (df['user_idx'], df['book_idx'])), shape=(n_users, n_books))

Total unique likes: 20155617
Total likes after filtering: 243190
Users: 10644
Books: 6906
Positive interactions: 243190
Sparsity: 99.6692%


## neighborhood-based

In [6]:
#1
likes_per_user = np.asarray(X.sum(axis=1)).ravel()
likes_per_book = np.asarray(X.sum(axis=0)).ravel()

print(f"Users: {n_users} | Books: {n_books} | Likes: {n_likes}")
print(f"Sparsity: {sparsity:.4%}")
print("\nLikes per user (min / mediaan / gemiddelde / max):", 
      likes_per_user.min(), np.median(likes_per_user), round(likes_per_user.mean(), 1), likes_per_user.max())
print("Likes per book (min / mediaan / gemiddelde / max):", 
      likes_per_book.min(), np.median(likes_per_book), round(likes_per_book.mean(), 1), likes_per_book.max())

Users: 10644 | Books: 6906 | Likes: 243190
Sparsity: 99.6692%

Likes per user (min / mediaan / gemiddelde / max): 1.0 18.0 22.8 254.0
Likes per book (min / mediaan / gemiddelde / max): 15.0 24.0 35.2 440.0


In [14]:
#2

# we transponeren de matrix zodat we de books kunnen vergelijken (die staan nu in de rijen, cosine_similarity werkt op rijen)
S = csr_matrix(cosine_similarity(X.T, dense_output=False))

print(f"Shape of similarity matrix: {S.shape}")
# Zet de diagonaal op 0 (een boek hoeft niet op zichzelf te lijken)
S.setdiag(0)
S.eliminate_zeros()

# Tel per boekenpaar hoe vaak ze SAMEN zijn geliket, en zet de diagonaal op 0
co_counts = (X.T @ X).tocsr()
co_counts.setdiag(0)

# Stel de drempelwaarde in en maak een kopie van de tabel voor de straffactor
beta = 10
shrinkage_weight = co_counts.copy()

# Bereken een weging tussen 0 en 1 (lage overlap = zware straf) en pas die toe op S
shrinkage_weight.data = shrinkage_weight.data / (shrinkage_weight.data + beta)
S_clean = S.multiply(shrinkage_weight)

print("X.T:")
print(X.T[:10, :10].toarray())

print("\nS_clean:")
print(S_clean[:10, :10].toarray().round(2))

Shape of similarity matrix: (6906, 6906)
X.T:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

S_clean:
[[0.   0.05 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.05 0.   0.04 0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.04 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.02 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.02 0.   0.   0.   0.   0.01]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.01 0.   0.   0.  ]]


In [ ]:
def recommend_books(user_idx, X, S_clean, K=20, top_n=5):

    # Maak een kopie van S_clean, zodat we de originele matrix niet overschrijven
    S_k = S_clean.tolil()

    # Loop door elk boek en bewaar alleen de K buren met de hoogste similarity-score (de rest wordt gewist)
    for i in range(S_k.shape[0]):
        if len(S_k.data[i]) > K:
            top_idx = np.sort(np.argsort(S_k.data[i])[-K:])
            S_k.data[i] = [S_k.data[i][j] for j in top_idx]
            S_k.rows[i] = [S_k.rows[i][j] for j in top_idx]

    # Pak alleen de rijen uit S_k van boeken die de user leuk vond, en tel ze per kolom op
    liked_books = X[user_idx].indices
    scores = np.asarray(S_k[liked_books].sum(axis=0)).ravel()

    # de boeken die de user al leuk vond, moeten we uitsluiten van de aanbevelingen met socore -1
    scores[X[user_idx].indices] = -1

    # sorteren van klein naar groot , omdraaien , en de eerste 5 terug geven
    top_books = np.argsort(scores)[::-1][:top_n]
    
    return top_books, scores[top_books]

# Test the influence of neighborhood size K on user 0
for K in [5, 20, 100]:
    books, scores = recommend_books(user_idx=0, X=X, S_clean=S_clean, K=K, top_n=5)
    print(f"K={K} -> Books: {books} | Scores: {np.round(scores, 2)}")

K=5 -> Books: [3765 4507 4039 2380 2428] | Scores: [0.81 0.62 0.49 0.48 0.45]
K=20 -> Books: [3765 4039 4507 4111 2380] | Scores: [0.81 0.69 0.62 0.52 0.48]
K=100 -> Books: [3765 4039 4507 5752 4111] | Scores: [0.9  0.85 0.72 0.68 0.65]


## model-based approach

In [ ]:
#1

def init_latent_vectors(X, d=20):
    # Haal het aantal users en boeken op en maak een random generator
    n_users, n_items = X.shape
    rng = np.random.default_rng(42)

    # Geef elke user (P) en elk boek (Q) een startvector van lengte d
    P = rng.normal(0, 0.1, size=(n_users, d))
    Q = rng.normal(0, 0.1, size=(n_items, d))
    
    return P, Q

def predict_scores(user_idx, P, Q):
    # Bereken het inproduct (p_u^T * q_i) tussen deze user en alle boeken
    return P[user_idx] @ Q.T

# Test Stap 1 (met nog ongetrainde vectoren)
P, Q = init_latent_vectors(X, d=20)
raw_scores = predict_scores(user_idx=0, P=P, Q=Q)

In [ ]:
#2

def train_als(X, P, Q, reg=0.1, alpha=40, iterations=10):
    # Zet de matrix ook klaar in kolom-vorm (X_T) en maak de regularization-matrix
    n_users, n_items = X.shape
    X_T = X.T.tocsr()
    I_d = np.eye(P.shape[1]) * reg

    # Wissel af: houd eerst Q vast om P te updaten, en houd daarna P vast om Q te updaten
    for _ in range(iterations):
        # 1. Update alle user-vectoren P
        QtQ = Q.T @ Q + I_d
        for u in range(n_users):
            items = X[u].indices
            Q_u = Q[items]
            A = QtQ + alpha * (Q_u.T @ Q_u)
            b = (1 + alpha) * Q_u.sum(axis=0)
            P[u] = np.linalg.solve(A, b)

        # 2. Update alle boek-vectoren Q
        PtP = P.T @ P + I_d
        for i in range(n_items):
            users = X_T[i].indices
            P_i = P[users]
            A = PtP + alpha * (P_i.T @ P_i)
            b = (1 + alpha) * P_i.sum(axis=0)
            Q[i] = np.linalg.solve(A, b)

    return P, Q

# Test Stap 2: train de vectoren P en Q op de interactiedata X
P, Q = train_als(X, P, Q, reg=0.1, alpha=40, iterations=10)

In [22]:
#3

def recommend_mf(user_idx, X, P, Q, top_n=5):
    # Bereken de match-scores met de functie uit stap 1
    scores = predict_scores(user_idx, P, Q)

    # Zet al gelezen boeken op -inf zodat ze niet opnieuw worden aangeraden
    scores[X[user_idx].indices] = -np.inf
    top_books = np.argsort(scores)[::-1][:top_n]
    
    return top_books, scores[top_books]

# Test de invloed van aantal factoren (d) en regularization (reg) op user 0
for d in [10, 50]:
    for reg in [0.01, 1.0]:
        P, Q = init_latent_vectors(X, d=d)
        P, Q = train_als(X, P, Q, reg=reg, iterations=10)
        books, scores = recommend_mf(user_idx=0, X=X, P=P, Q=Q, top_n=5)
        print(f"d={d}, reg={reg} -> Books: {books} | Scores: {np.round(scores, 2)}")

d=10, reg=0.01 -> Books: [4401 4212 4039 4077 4602] | Scores: [0.76 0.76 0.74 0.7  0.7 ]
d=10, reg=1.0 -> Books: [4401 4212 4039 4077 4602] | Scores: [0.76 0.76 0.74 0.7  0.69]


KeyboardInterrupt: 